# Document Intelligence vs. Content Understanding for claim extraction

This notebook runs the same synthetic two-page claim form through two schema-based extraction approaches:

1. **Azure AI Document Intelligence** `prebuilt-layout` with the premium **Query Fields** add-on.
2. **Azure Content Understanding** with a custom `extract` field schema.

It compares reviewed-field accuracy, extraction coverage, service latency, optional Azure AI Language PII latency, current regional list-price components, rate limits, and a downstream Azure AI Search index-size scenario. Content Understanding analyzer setup time is reported separately from per-document analysis latency.

> PII detection is a separate Azure AI Language stage. It is not assumed to be included in either field-extraction approach.

## 1. Environment and configuration

The notebook loads the repository-root `.env`, uses `DefaultAzureCredential`, and refuses to execute from an interpreter other than the repository `.venv`. Live calls are opt-in because they incur charges.

In [30]:
from pathlib import Path
import importlib
import os
import sys

import pandas as pd
from IPython.display import Markdown, display

search_roots = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(
    (root for root in search_roots if (root / "demo" / "extraction-comparison").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the azure-ai-foundry-insurance repository.")

NOTEBOOK_DIR = REPO_ROOT / "demo" / "extraction-comparison" / "notebooks"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

importlib.invalidate_caches()
import _lib as comparison_lib
comparison_lib = importlib.reload(comparison_lib)

from _lib import (
    DEFAULT_DOCUMENT,
    REPORT_PATH,
    accuracy_summary,
    build_rate_card,
    compare_fields,
    configure_content_understanding_defaults,
    ensure_content_understanding_analyzer,
    env_flag,
    extract_markdown,
    get_default_credential,
    infer_pricing_region,
    index_size_rows,
    latency_rows,
    load_environment,
    load_ground_truth,
    make_content_understanding_client,
    make_document_intelligence_client,
    observed_rate_headers,
    pricing_rows,
    pricing_totals,
    rate_card_rows,
    rate_limit_rows,
    require_repo_venv,
    resolve_pricing_region,
    run_content_understanding,
    run_document_intelligence,
    run_text_pii,
    throughput_rows,
    write_comparison_report,
)

load_environment()
PYTHON = require_repo_venv()

DOCUMENT_PATH = DEFAULT_DOCUMENT
ANALYZER_ID = os.getenv(
    "EXTRACTION_COMPARISON_ANALYZER_ID",
    "claimExtractionComparisonV1",
)
RUN_AZURE = env_flag("EXTRACTION_COMPARISON_RUN_AZURE", False)
RUN_PII = env_flag("EXTRACTION_COMPARISON_RUN_PII", False)
CONFIGURE_DEFAULTS = env_flag("EXTRACTION_COMPARISON_CONFIGURE_DEFAULTS", True)
PRICING_REGION = (os.getenv("AZURE_PRICING_REGION") or "").strip() or None
LANGUAGE_TIER = os.getenv("EXTRACTION_COMPARISON_LANGUAGE_TIER", "S")
CORPUS_DOCUMENTS = int(os.getenv("EXTRACTION_COMPARISON_CORPUS_DOCUMENTS", "1000000"))

assert DOCUMENT_PATH.is_file(), f"Missing sample document: {DOCUMENT_PATH}"
print({
    "python": str(PYTHON),
    "document": str(DOCUMENT_PATH.relative_to(REPO_ROOT)),
    "run_azure": RUN_AZURE,
    "run_pii": RUN_PII,
    "pricing_region": PRICING_REGION or "derive from live x-ms-region header",
    "projected_documents": CORPUS_DOCUMENTS,
})

{'python': 'C:\\Users\\jomedin\\Documents\\azure-ai-foundry-insurance\\.venv\\Scripts\\python.exe', 'document': 'data\\propertyclaims\\ClaimForm_1.pdf', 'run_azure': False, 'run_pii': False, 'pricing_region': 'derive from live x-ms-region header', 'projected_documents': 1000000}


## 2. Published limits and throughput caveats

These are admission and payload limits, not completion-throughput guarantees. Real throughput is also bounded by document complexity, model TPM, retries, and downstream indexing.

In [31]:
limits = pd.DataFrame(rate_limit_rows(LANGUAGE_TIER))
display(limits)

,Service,Admission limit,Payload limit,Adjustment,Source
0,Document Intelligence S0,15 analyze POST/s; 50 result GET/s,"500 MB; 2,000 pages",TPS can be raised by support request,https://learn.microsoft.com/azure/ai-services/...
1,Content Understanding S0,"1,000 pages/images/min; 3,000 operations/min",Async: 200 MB and 300 pages,Also constrained by Foundry model TPM,https://learn.microsoft.com/azure/ai-services/...
2,Azure AI Language PII S,"1,000 requests/s and 1,000 requests/min","Sync: 5 documents/request and 5,120 chars/docu...",Chunk before submission; retry 429 responses,https://learn.microsoft.com/azure/ai-services/...
3,Azure AI Search push indexing,Batch API limits apply,"1,000 documents or 16 MB/request",Benchmark SKU; partitions help indexing capacity,https://learn.microsoft.com/azure/search/searc...


## 3. Clients and Content Understanding analyzer

Analyzer deployment is idempotent and owner-guarded. Its setup latency is excluded from per-document latency because production systems create analyzers ahead of ingestion.

In [32]:
credential = None
di_client = None
cu_client = None
cu_setup_seconds = None
cu_setup_observations = []

if RUN_AZURE:
    credential = get_default_credential()
    di_client = make_document_intelligence_client(credential)
    cu_client = make_content_understanding_client(credential)
    if CONFIGURE_DEFAULTS:
        deployments = configure_content_understanding_defaults(credential)
        print("Configured CU model aliases:", sorted(deployments))
    analyzer, cu_setup_seconds, cu_setup_observations = ensure_content_understanding_analyzer(
        cu_client,
        ANALYZER_ID,
    )
    print({
        "analyzer_id": ANALYZER_ID,
        "status": analyzer.get("status", "ready"),
        "setup_seconds": round(cu_setup_seconds, 3),
    })
else:
    print("Dry run only. Set EXTRACTION_COMPARISON_RUN_AZURE=true in .env for paid calls.")

Dry run only. Set EXTRACTION_COMPARISON_RUN_AZURE=true in .env for paid calls.


## 4. Execute both extraction approaches

Both calls receive the same file and the same 14 logical field names. The DI call uses Query Fields with text output; the CU call uses descriptions and types from the analyzer schema. Text output is intentional because this resource omits the Query Fields `documents` collection when Markdown output is requested in the same DI call.

In [33]:
di_run = None
cu_run = None

if RUN_AZURE:
    di_run = run_document_intelligence(di_client, DOCUMENT_PATH)
    cu_run = run_content_understanding(cu_client, ANALYZER_ID, DOCUMENT_PATH)
    print({
        "di_seconds": round(di_run.latency_seconds, 3),
        "cu_seconds": round(cu_run.latency_seconds, 3),
        "cu_usage": cu_run.usage,
    })
else:
    print("Extraction skipped in dry-run mode.")

Extraction skipped in dry-run mode.


## 5. Field quality

This is normalized exact-match scoring against reviewed values. It is stricter and more useful than comparing confidence scores alone.

In [34]:
comparison = None
if di_run and cu_run:
    comparison_rows = compare_fields(load_ground_truth(), di_run.payload, cu_run.payload)
    comparison = pd.DataFrame(comparison_rows)
    display(comparison)
    display(pd.DataFrame(accuracy_summary(comparison_rows)))
else:
    print("Run the Azure calls to populate the field comparison.")

Run the Azure calls to populate the field comparison.


## 6. Optional PII stage

Azure AI Language PII is applied separately to the extracted text from each approach. Text is split below the synchronous 5,120-character limit and submitted in batches of at most five documents. The notebook counts billable 1,000-character text records and captures throttling headers. Detection does not prove that every sensitive value was found; inspect false negatives before indexing.

In [35]:
di_text = extract_markdown(di_run.payload, "Document Intelligence") if di_run else ""
cu_text = extract_markdown(cu_run.payload, "Content Understanding") if cu_run else ""
di_pii_run = None
cu_pii_run = None

if RUN_AZURE and RUN_PII:
    di_pii_run = run_text_pii(di_text, credential, label="DI text")
    cu_pii_run = run_text_pii(cu_text, credential, label="CU text")
    display(pd.DataFrame([
        {
            "Approach": "Document Intelligence",
            "Characters": di_pii_run.usage["characters"],
            "Billable text records": di_pii_run.usage["text_records"],
            "PII entities": di_pii_run.payload["entity_count"],
            "PII seconds": di_pii_run.latency_seconds,
        },
        {
            "Approach": "Content Understanding",
            "Characters": cu_pii_run.usage["characters"],
            "Billable text records": cu_pii_run.usage["text_records"],
            "PII entities": cu_pii_run.payload["entity_count"],
            "PII seconds": cu_pii_run.latency_seconds,
        },
    ]))
elif RUN_AZURE:
    print("PII skipped. Set EXTRACTION_COMPARISON_RUN_PII=true to include it.")
else:
    print("PII skipped in dry-run mode.")

PII skipped in dry-run mode.


## 7. Latency and observed rate-limit signals

A single execution is a functional sample, not a latency benchmark. For a decision, run a representative corpus and report warm p50, p95, p99, throttling rate, queue age, and time-to-searchability.

In [36]:
if di_run and cu_run:
    display(pd.DataFrame(latency_rows(di_run, cu_run, di_pii_run, cu_pii_run)))
    page_count = len(di_run.payload.get("pages") or [])
    display(pd.DataFrame(throughput_rows(
        page_count,
        di_run,
        cu_run,
        di_pii_run,
        LANGUAGE_TIER,
    )))
    observed = observed_rate_headers(di_run, cu_run, di_pii_run, cu_pii_run)
    if cu_setup_observations:
        observed = [*cu_setup_observations, *observed]
    display(pd.DataFrame(observed))
else:
    print("Run the Azure calls to populate observed latency and headers.")

Run the Azure calls to populate observed latency and headers.


## 8. Pricing comparison

The rate card is fetched from the public Azure Retail Prices API. Region selection uses `AZURE_PRICING_REGION` when configured, then the live Azure `x-ms-region` response header, and finally `eastus` for a nonbillable fallback. The DI estimate adds the base prebuilt/layout meter and Query Fields add-on. The CU estimate uses the response's page, contextualization, model-input, cached-input, and model-output usage. Optional PII is added independently to each path.

These are retail list-price estimates. Verify meter selection, deployment geography (`Global` versus `Data Zone`), negotiated discounts, commitment tiers, and the Azure bill before making a production decision.

In [37]:
effective_pricing_region, pricing_region_source = resolve_pricing_region(
    PRICING_REGION,
    di_run,
    cu_run,
)
print(f"Retail pricing region: {effective_pricing_region} ({pricing_region_source})")
rate_card = build_rate_card(effective_pricing_region)
display(pd.DataFrame(rate_card_rows(rate_card)))

if di_run and cu_run:
    component_costs = pricing_rows(
        rate_card,
        di_run,
        cu_run,
        di_pii_run,
        cu_pii_run,
    )
    display(pd.DataFrame(component_costs))
    display(pd.DataFrame(pricing_totals(component_costs)))
else:
    print("Current rates are shown above; measured usage is required for per-document totals.")

Retail pricing region: eastus (default fallback)


,Key,Product,Meter,USD,Unit,Effective
0,di_layout,Azure Document Intelligence,S0 Pre-built Pages,10.000,1K,2021-08-01T00:00:00Z
1,di_query_fields,Azure Document Intelligence,S0 pages for query fields,10.000,1K,2024-11-01T00:00:00Z
2,language_pii,Azure Language,Standard Text Records,1.000,1K,2020-12-01T00:00:00Z
3,cu_minimal,Azure Content Understanding,Doc Content Extraction Min Pages,0.010,1K,2026-01-01T00:00:00Z
4,cu_basic,Azure Content Understanding,Doc Content Extraction Basic Pages,1.000,1K,2025-12-01T00:00:00Z
5,cu_standard,Azure Content Understanding,Doc Content Extraction Standard Pages,5.000,1K,2025-12-01T00:00:00Z
6,cu_contextualization,Azure Content Understanding,Std Contextualization Tokens,0.001,1K,2025-12-01T00:00:00Z
7,cu_model_input,Azure OpenAI GPT5,GPT 5.2 inp Gl 1M Tokens,1.750,1M,2025-12-01T00:00:00Z
8,cu_model_cached_input,Azure OpenAI GPT5,GPT 5.2 cd inp Gl 1M Tokens,0.175,1M,2025-12-01T00:00:00Z
9,cu_model_output,Azure OpenAI GPT5,GPT 5.2 opt Gl 1M Tokens,14.000,1M,2025-12-01T00:00:00Z


Current rates are shown above; measured usage is required for per-document totals.


## 9. Downstream index-size scenario

This is a planning estimate, not an Azure AI Search quota calculation. Raw vector bytes are deterministic; the 1.3 HNSW multiplier is an explicit scenario assumption. Search also stores text, filterable metadata, internal structures, and temporary blue/green indexes. Validate actual usage with Search service and index statistics.

In [38]:
if di_text and cu_text:
    index_projection = pd.DataFrame(index_size_rows(
        di_text,
        cu_text,
        corpus_documents=CORPUS_DOCUMENTS,
        chunk_characters=2_000,
        overlap_characters=200,
        embedding_dimensions=3_072,
        bytes_per_dimension=4,
        hnsw_scenario_multiplier=1.3,
    ))
    display(index_projection)
else:
    print("Run extraction to estimate downstream chunk and vector size.")

Run extraction to estimate downstream chunk and vector size.


## 10. Final consolidated report

A live run consolidates field quality, latency, PII, pricing, rate limits, throughput, and index sizing into `demo/extraction-comparison/REPORT.md`. It also writes Markdown and JSON snapshots under the ignored `output/` directory.

In [39]:
if di_run and cu_run:
    report_comparison_rows = compare_fields(
        load_ground_truth(),
        di_run.payload,
        cu_run.payload,
    )
    report_latency_data = latency_rows(
        di_run,
        cu_run,
        di_pii_run,
        cu_pii_run,
    )
    report_page_count = len(di_run.payload.get("pages") or [])
    report_throughput_data = throughput_rows(
        report_page_count,
        di_run,
        cu_run,
        di_pii_run,
        LANGUAGE_TIER,
    )
    report_region, report_region_source = resolve_pricing_region(
        PRICING_REGION,
        di_run,
        cu_run,
    )
    report_rate_card = build_rate_card(report_region)
    report_price_components = pricing_rows(
        report_rate_card,
        di_run,
        cu_run,
        di_pii_run,
        cu_pii_run,
    )
    report_index_data = index_size_rows(
        di_text,
        cu_text,
        corpus_documents=CORPUS_DOCUMENTS,
        chunk_characters=2_000,
        overlap_characters=200,
        embedding_dimensions=3_072,
        bytes_per_dimension=4,
        hnsw_scenario_multiplier=1.3,
    )
    final_report = write_comparison_report(
        document_path=DOCUMENT_PATH.relative_to(REPO_ROOT),
        comparison_rows=report_comparison_rows,
        latency_data=report_latency_data,
        throughput_data=report_throughput_data,
        price_component_rows=report_price_components,
        index_data=report_index_data,
        rate_limits=rate_limit_rows(LANGUAGE_TIER),
        cu_usage=cu_run.usage,
        pricing_region=report_region,
        pricing_region_source=report_region_source,
        cu_setup_seconds=cu_setup_seconds,
        di_run=di_run,
        cu_run=cu_run,
        di_pii_run=di_pii_run,
        cu_pii_run=cu_pii_run,
    )
    display(Markdown(final_report["markdown"]))
    print("Report:", final_report["report_path"])
    print("JSON:", final_report["json_path"])
else:
    print(
        "Live report not regenerated because Azure extraction did not run. "
        f"Open the latest verified report at {REPORT_PATH}."
    )

Live report not regenerated because Azure extraction did not run. Open the latest verified report at C:\Users\jomedin\Documents\azure-ai-foundry-insurance\demo\extraction-comparison\REPORT.md.


## 11. Decision checklist

Choose only after testing a representative corpus—not this single clean form.

- **Quality:** field precision/recall, table fidelity, handwriting, confidence calibration, and human-review rate.
- **Cost:** extraction, Query Fields, PII, CU contextualization, model tokens, retries, reprocessing, and Search capacity.
- **Latency:** warm p50/p95/p99, backfill completion, queue age, and time until content is searchable.
- **Rate limits:** DI POST/GET quotas, CU pages/minute, Language batching, model TPM, and Search indexing limits.
- **Privacy:** whether text is merely detected or actually redacted before embeddings, Search, logs, and traces.
- **Operations:** idempotency, dead-letter replay, versioned schemas/models/indexes, and deletion propagation.

**Likely trade-off:** Document Intelligence is generally the more deterministic and economical option for stable forms. Content Understanding is generally more flexible for varied or semantically complex documents, but its contextualization/model usage and latency must be measured.